In [30]:
!git clone https://github.com/expaetra/CM3070_final_project.git
%cd CM3070_final_project
!git lfs install
!git lfs pull

Cloning into 'CM3070_final_project'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: 'CM3070_final_project'
/content
Git LFS initialized.
Not in a git repository.


## Modeling on expanded data: titles + abstract

After collecting the abstract along with their pertaining paper titles, the same classifier is used to assess whether whether incorporating titles, alongside the increased dataset size leads to further performance improvement.

In [31]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.feature_extraction.text import TfidfVectorizer

import nltk
from nltk.corpus import stopwords

In [32]:
# load dataset

df = pd.read_csv("/content/arxiv_abstracts_and_titles.csv")
print(df.shape)
df.head()

(79141, 7)


,category,field,discipline,title,abstract,abstract_length,title_length
0,cs.LG,Machine Learning,Artificial Intelligence,how do lexical semantics affect translation an...,neural machine translation (nmt) systems aim t...,177,9
1,cs.LG,Machine Learning,Artificial Intelligence,barack partially supervised group robustness w...,while neural networks have shown remarkable su...,203,7
2,cs.LG,Machine Learning,Artificial Intelligence,a deep learning approach to integrate humanlev...,"in recent times, a large number of people have...",187,11
3,cs.LG,Machine Learning,Artificial Intelligence,croesus multistage processing and transactions...,emerging edge applications require both a fast...,194,10
4,cs.LG,Machine Learning,Artificial Intelligence,representation topology divergence a method fo...,comparison of data representations is a comple...,133,10


In [44]:
# show distribution

discipline_counts = df['discipline'].value_counts()
field_counts = df['field'].value_counts()
print("Discipline:\n", discipline_counts)
print("\nField:\n", field_counts)

Discipline:
 discipline
Artificial Intelligence         17464
Theory of Computation            7497
Multimedia                       5000
World Wide Web                   4754
Communication                    2500
Computer Imaging and Vision      2500
Information Retrieval            2500
Cybersecurity                    2500
Robotics                         2500
Computer Networks                2500
Computer Graphics                2500
Computer Hardware                2500
Database Systems                 2500
Human-Computer Interaction       2499
Social Computing                 2499
Computational Science            2499
Distributed Systems              2498
Software Engineering             2494
Computer Programming             2486
Emerging Technologies            2474
Theoretical Computer Science     2301
Computer Systems                 2176
Name: count, dtype: int64

Field:
 field
Machine Learning                        2500
Computer Vision                         2500
General A

In [45]:
# apply cap

cap = 2500

df_capped = (
    df.groupby('discipline', group_keys=False)[df.columns]
      .apply(lambda x: x.sample(n=min(len(x), cap), random_state=42))
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)

print(df_capped['discipline'].value_counts())

discipline
Information Retrieval           2500
Multimedia                      2500
Computer Graphics               2500
Computer Imaging and Vision     2500
Cybersecurity                   2500
Theory of Computation           2500
Communication                   2500
Database Systems                2500
Artificial Intelligence         2500
Computer Hardware               2500
World Wide Web                  2500
Robotics                        2500
Computer Networks               2500
Human-Computer Interaction      2499
Computational Science           2499
Social Computing                2499
Distributed Systems             2498
Software Engineering            2494
Computer Programming            2486
Emerging Technologies           2474
Theoretical Computer Science    2301
Computer Systems                2176
Name: count, dtype: int64


In [46]:
# apply cap

cap = 2500

# combine title + abstract

df_capped["text"] = df_capped["title"] + " " +df_capped["abstract"]
df["text"] = df["title"] + " " + df["abstract"]

# discipline
X_text_disc = df_capped["text"].astype(str)
y_disc = df_capped["discipline"]

# field
X_text_field = df["text"].astype(str)
y_field = df["field"]

In [47]:
# stopwords

nltk.download("stopwords")

custom_stops = {
    'based', 'paper', 'show', 'results', 'problem', 'using', 'approach',
    'proposed', 'method', 'methods', 'propose', 'present', 'work',
    'used', 'use', 'two', 'one', 'new', 'also', 'shows', 'however',
    'provide', 'study', 'task', 'tasks', 'different', 'high', 'given'
}
stop_words = set(stopwords.words('english')).union(custom_stops)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [48]:
# prepare data

# discipline (capped)
X_text_disc = df_capped["abstract"].astype(str)
y_disc = df_capped["discipline"]

# field
X_text_field = df["abstract"].astype(str)
y_field = df["field"]

In [49]:
# train/test split

# discipline
X_train_text_disc, X_test_text_disc, y_disc_train, y_disc_test = train_test_split(
    X_text_disc,
    y_disc,
    test_size=0.2,
    random_state=42,
    stratify=y_disc
)

# field
X_train_text_field, X_test_text_field, y_field_train, y_field_test = train_test_split(
    X_text_field,
    y_field,
    test_size= 0.2,
    random_state=42,
    stratify=y_field
)

In [50]:
# TF-IDF (best representation from earlier experiments)

# TF-IDF (same setup for both)

vectorizer_disc = TfidfVectorizer(
    stop_words=list(stop_words),
    ngram_range=(1, 1),
    max_features=5000
)

vectorizer_field =TfidfVectorizer(
    stop_words=list(stop_words),
    ngram_range=(1,1),
    max_features=5000
)

# fit + transform
X_train_disc = vectorizer_disc.fit_transform(X_train_text_disc)
X_test_disc = vectorizer_disc.transform(X_test_text_disc)

X_train_field = vectorizer_field.fit_transform(X_train_text_field)
X_test_field =vectorizer_field.transform(X_test_text_field)

In [51]:
def evaluate_result(y_test, y_pred, task, model_name, representation):

    accuracy = accuracy_score(y_test, y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average='macro', zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average='weighted',zero_division=0
    )

    return {
        "Task": task,
        "Model": model_name,
        "Representation": representation,
        "Accuracy": round(accuracy, 3),
        "Macro Precision": round(macro_p,3),
        "Macro Recall": round(macro_r, 3),
        "Macro F1": round(macro_f1, 3),
        "Weighted Precision": round(weighted_p, 3),
        "Weighted Recall": round(weighted_r, 3),
        "Weighted F1": round(weighted_f1, 3)
    }

In [52]:
# train the logistic regression models

disc_model = LogisticRegression(max_iter=1000, random_state=42)
disc_model.fit(X_train_disc, y_disc_train)

field_model = LogisticRegression(max_iter=1000,random_state=42)
field_model.fit(X_train_field, y_field_train)

LogisticRegression(max_iter=1000, random_state=42)

In [53]:
# predictions
y_disc_pred = disc_model.predict(X_test_disc)
y_field_pred = field_model.predict(X_test_field)

In [54]:
results = []

results.append(
    evaluate_result(
        y_disc_test,
        y_disc_pred,
        task="Discipline",
        model_name="Logistic Regression",
        representation="TF-IDF word unigrams (capped discipline)"
    )
)
results.append(
    evaluate_result(
        y_field_test,
        y_field_pred,
        task="Field",
        model_name="Logistic Regression",
        representation="TF-IDF word unigrams (uncapped field)"
    )
)

results_df = pd.DataFrame(results)
results_df

,Task,Model,Representation,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
0,Discipline,Logistic Regression,TF-IDF word unigrams (capped discipline),0.614,0.611,0.612,0.610,0.611,0.614,0.611
1,Field,Logistic Regression,TF-IDF word unigrams (uncapped field),0.527,0.519,0.528,0.522,0.518,0.527,0.521


In [55]:
# save

os.makedirs("/content/drive/MyDrive/CM3070_final_project/backend/data", exist_ok=True)
df_capped.to_csv(
    "/content/drive/MyDrive/CM3070_final_project/backend/data/abstracts_and_title_modeling.csv",
    index=False
)

results_df.to_csv(
    "/content/drive/MyDrive/CM3070_final_project/backend/data/abstracts_and_title_modeling_results.csv",
    index=False
)

print("Saved")

Saved
